In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.subplots as sp
import plotly.graph_objects as go
import random

In [ ]:
def load_session_data(session_path, signals_to_plot):
    """
    Load session data for the selected physiological signals.
    Returns a dictionary with signal name as key and its DataFrame as value.
    """
    data = {}
    for signal in signals_to_plot:
        signal_path = os.path.join(session_path, f"{signal}.csv")
        if os.path.exists(signal_path):
            data[signal] = pd.read_csv(signal_path)
    return data

def get_random_session(raw_data_path):
    """
    Select a random session from the directory structure.
    """
    sessions = []
    for root, dirs, _ in os.walk(raw_data_path):
        for dir_name in dirs:
            if '_' in dir_name and dir_name.count('_') == 2:
                sessions.append(os.path.join(root, dir_name))
    return random.choice(sessions) if sessions else None


In [ ]:
# Set the raw data path and signals to plot
RAW_DATA_PATH = "data/raw"
to_plot = ['eda', 'bvp', 'temperature', "accelerometer"]

# Get a random session
random_session = get_random_session(RAW_DATA_PATH)
if not random_session:
    raise ValueError("No valid sessions found in the specified directory structure.")

# Load data for the selected session
session_data = load_session_data(random_session, to_plot)

# Load tracks.csv to get song start and end times
tracks_path = os.path.join(random_session, "tracks.csv")
tracks = pd.read_csv(tracks_path) if os.path.exists(tracks_path) else None


In [ ]:
# Create a subplot layout for the signals
fig = sp.make_subplots(rows=len(session_data), cols=1, shared_xaxes=True, 
                       subplot_titles=to_plot)

# Iterate through signals and add traces
for i, (signal, df) in enumerate(session_data.items(), start=1):
    if 'unix_timestamp' in df.columns:
        x = pd.to_datetime(df['unix_timestamp'], unit='us')  # Convert to datetime if needed
    else:
        x = np.arange(len(df))  # Fallback to sequential index
    if signal == "accelerometer":
        for axis in ['x', 'y', 'z']:
            y = df[axis]
            fig.add_trace(go.Scatter(x=x, y=y, name=f'Accelerometer ({axis})'), row=i, col=1)
    else:
        y = df.iloc[:, 1]  # Assuming the signal value is in the second column
        fig.add_trace(go.Scatter(x=x, y=y, name=signal.capitalize()), row=i, col=1)

colour_mapping = {"Affective":"lightblue",
                  "Eudaimonic":"lightsalmon",
                  "Goal-Attainment":"lightgreen",
                  "Other":"lightcoral"}

majority_context = tracks["context"].value_counts().sort_values(ascending=False).idxmax()

# Highlight session duration and add track end markers
if tracks is not None:
    session_start = pd.to_datetime(tracks['started_at'].min(), unit='us')
    session_end = pd.to_datetime(tracks['ended_at'].max(), unit='us')
    
    # Add a vrect for the entire session
    fig.add_vrect(
        x0=session_start, x1=session_end,
        fillcolor=colour_mapping[majority_context], opacity=0.4, line_width=0
    )
    
    # Add vertical lines for each track's end, except the last one
    for i, track in tracks.iterrows():
        # if i < len(tracks) - 1:  # Skip the last track
        fig.add_vline(
            x=track["ended_at"]/1000,
            line_dash="dash",
            line_color="black",
            annotation_text=f"{track['track_name']}", annotation_position="top left",
            annotation_font_size=6
        )

# Update layout
fig.update_layout(height=150*len(session_data), title=f"Session: {random_session} | {majority_context}")
fig.show()
